In [1]:
!pip -q install -U "transformers>=4.41.0" accelerate peft trl datasets sacrebleu sentencepiece bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.6 MB/s eta 0:00:00


In [2]:
import os
import re
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from sacrebleu.metrics import BLEU

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
intel_path = "drive/My Drive/cs263_final_proj/intel_misclassified_top200.csv"
pubhealth_path = "drive/My Drive/cs263_final_proj/pubhealth_misclassified_top200.csv"

intel_df = pd.read_csv(intel_path)
pub_df = pd.read_csv(pubhealth_path)

print("Intel cols:", list(intel_df.columns))
print("PubHealth cols:", list(pub_df.columns))

display(intel_df.head(2))
display(pub_df.head(2))

Intel cols: ['text', 'reasoning', 'label', 'model', 'label_text', 'labels', 'idx', 'true_id', 'true_label', 'pred_id', 'pred_label', 'pred_confidence', 'prob_true', 'prob_false', 'prob_mixture', 'model_input_text']
PubHealth cols: ['claim_id', 'claim', 'date_published', 'explanation', 'fact_checkers', 'main_text', 'sources', 'labels', 'subjects', 'idx', 'true_id', 'true_label', 'pred_id', 'pred_label', 'pred_confidence', 'prob_true', 'prob_false', 'prob_mixture', 'model_input_text']


,text,reasoning,label,model,label_text,labels,idx,true_id,true_label,pred_id,pred_label,pred_confidence,prob_true,prob_false,prob_mixture,model_input_text
0,Studies have consistently shown that social me...,This is false because numerous studies have de...,0,meta-llama/Meta-Llama-3.1-8B-Instruct,false,1,3200,1,false,0,true,0.998243,0.998243,0.000421,0.001336,[CLAIM] Studies have consistently shown that s...
1,AI systems will eventually surpass human intel...,This statement contains some truth because AI ...,2,meta-llama/Meta-Llama-3.1-8B-Instruct,partially_true,2,748,2,mixture,1,false,0.997742,0.000470,0.997742,0.001788,[CLAIM] AI systems will eventually surpass hum...


,claim_id,claim,date_published,explanation,fact_checkers,main_text,sources,labels,subjects,idx,true_id,true_label,pred_id,pred_label,pred_confidence,prob_true,prob_false,prob_mixture,model_input_text
0,9975,Crohn’s patients respond to J&J’s Stelara in s...,"May 8, 2011","Without evaluating the evidence, it becomes on...",NaN,"While we’re given the size of the market, we d...",NaN,2,Reuters Health,1165,2,mixture,0,true,0.997357,0.997357,0.000357,0.002286,[CLAIM] Crohn’s patients respond to J&J’s Stel...
1,30893,A teenage schoolgirl in Texas became pregnant ...,"October 5, 2015",WNDR assumes however all responsibility for th...,Kim LaCapria,A 14-year old schoolgirl has suffered serious ...,NaN,1,"Junk News, flu shot, world news daily report",67,1,false,0,true,0.997139,0.997139,0.000672,0.002190,[CLAIM] A teenage schoolgirl in Texas became p...


In [6]:
def build_explanation_prompt(claim, predicted_label, confidence, evidence=None):
    evidence_block = ""
    if isinstance(evidence, str) and evidence.strip():
        evidence_block = f"\nEvidence (provided): {evidence.strip()[:800]}"

    return f"""You are generating a short, faithful explanation for a misinformation classifier.

Rules:
- Output EXACTLY 1–2 sentences.
- Do NOT add external facts.
- Only use the claim and the provided evidence (if any).
- Must be consistent with the predicted label.
- If label is 'mixture' or 'questionable', emphasize mixed/insufficient evidence and uncertainty.
- Reuse important keywords from the claim/evidence to match dataset style.

Claim: {claim}
Predicted label: {predicted_label}
Model confidence: {float(confidence):.2f}{evidence_block}

Explanation:"""

In [7]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def row_to_text(claim, pred_label, pred_conf, gold_expl, evidence=None):
    prompt = build_explanation_prompt(claim, pred_label, pred_conf, evidence=evidence)
    messages = [
        {"role": "system", "content": "You write short, faithful classifier explanations."},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": str(gold_expl).strip()},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def build_dataset_from_df(
    df,
    claim_col,
    gold_col,
    label_col="pred_label",
    conf_col="pred_confidence",
    evidence_col=None,
    max_rows=200,
):
    records = []
    df = df.head(max_rows).copy()

    for _, r in df.iterrows():
        claim = r.get(claim_col, "")
        gold  = r.get(gold_col, "")
        if not isinstance(claim, str) or not claim.strip():
            continue
        if not isinstance(gold, str) or not gold.strip():
            continue

        pred_label = r.get(label_col, "mixture")
        pred_conf  = r.get(conf_col, 0.5)
        try:
            pred_conf = float(pred_conf)
        except:
            pred_conf = 0.5

        evidence = r.get(evidence_col, None) if evidence_col else None
        text = row_to_text(claim, pred_label, pred_conf, gold, evidence=evidence)
        records.append({"text": text})

    return Dataset.from_list(records)

# Intel: text + reasoning
intel_ds = build_dataset_from_df(
    intel_df,
    claim_col="text",
    gold_col="reasoning",
    label_col="pred_label",
    conf_col="pred_confidence",
    evidence_col=None,
    max_rows=200
)

# PubHealth: claim + explanation + main_text
pub_ds = build_dataset_from_df(
    pub_df,
    claim_col="claim",
    gold_col="explanation",
    label_col="pred_label",
    conf_col="pred_confidence",
    evidence_col="main_text",
    max_rows=200
)

full_ds = Dataset.from_list(intel_ds.to_list() + pub_ds.to_list())
print("Total SFT examples:", len(full_ds))

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Total SFT examples: 400


In [9]:
split = full_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [14]:
out_dir = "drive/My Drive/cs263_final_proj/data/llama32_expl_sft_lora"

sft_args = SFTConfig(
    output_dir=out_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
    packing=False,  # keep each example separate
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)

trainer.train()

# Save adapter + tokenizer
trainer.save_model(os.path.join(out_dir, "final"))
tokenizer.save_pretrained(os.path.join(out_dir, "final"))

print("Saved fine-tuned LoRA adapter to:", os.path.join(out_dir, "final"))

Adding EOS to train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
10,3.029988
20,2.236110
30,1.606157
40,1.605204
50,1.504825
60,1.373446
70,1.442346
80,1.525456
90,1.517231
100,1.339231


Saved fine-tuned LoRA adapter to: drive/My Drive/cs263_final_proj/data/llama32_expl_sft_lora/final


In [15]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BLEU_METRIC = BLEU()

FT_DIR = "drive/My Drive/cs263_final_proj/data/llama32_expl_sft_lora/final"

# Load base + LoRA adapter
base = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
ft_model = PeftModel.from_pretrained(base, FT_DIR)
ft_model.eval()

ft_tokenizer = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

def extract_assistant_text(decoded: str) -> str:
    s = decoded.strip()
    if "assistant" in s:
        s = s.split("assistant")[-1].strip()
    s = re.sub(r'^(system|user)\s*', '', s, flags=re.IGNORECASE).strip()
    return s

def to_two_sentences_str(text: str) -> str:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    parts = [p.strip() for p in parts if p.strip()]
    return " ".join(parts[:2])

@torch.no_grad()
def generate_expl_ft(prompt: str, max_new_tokens=80) -> str:
    messages = [
        {"role": "system", "content": "You write short, faithful classifier explanations."},
        {"role": "user", "content": prompt},
    ]
    input_text = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ft_tokenizer(input_text, return_tensors="pt").to(ft_model.device)

    out = ft_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        repetition_penalty=1.05,
        pad_token_id=ft_tokenizer.eos_token_id,
        eos_token_id=ft_tokenizer.eos_token_id,
    )
    decoded = ft_tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(input_text):
        decoded = decoded[len(input_text):].strip()
    decoded = extract_assistant_text(decoded)
    return to_two_sentences_str(decoded)

def bleu_eval_df(df, dataset_name, claim_col, gold_col, evidence_col=None, n=200):
    df = df.head(n).copy()
    preds, refs = [], []

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"BLEU eval {dataset_name}"):
        claim = r.get(claim_col, "")
        gold  = r.get(gold_col, "")
        if not isinstance(claim, str) or not claim.strip():
            continue
        if not isinstance(gold, str) or not gold.strip():
            continue

        pred_label = r.get("pred_label", "mixture")
        pred_conf  = r.get("pred_confidence", 0.5)
        try:
            pred_conf = float(pred_conf)
        except:
            pred_conf = 0.5

        evidence = r.get(evidence_col, None) if evidence_col else None
        prompt = build_explanation_prompt(claim, pred_label, pred_conf, evidence=evidence)

        gen = generate_expl_ft(prompt, max_new_tokens=80)
        preds.append(gen)
        refs.append(gold)

    score = BLEU_METRIC.corpus_score(preds, [refs]).score
    print(f"[{dataset_name}] scored={len(preds)} BLEU={score:.2f}")
    return score

intel_bleu = bleu_eval_df(intel_df, "intel_misclassified_top200", "text", "reasoning", None, n=200)
pub_bleu   = bleu_eval_df(pub_df, "pubhealth_misclassified_top200", "claim", "explanation", "main_text", n=200)

print("BLEU after SFT:")
print("Intel:", intel_bleu)
print("PubHealth:", pub_bleu)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

BLEU eval intel_misclassified_top200:   0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[intel_misclassified_top200] scored=200 BLEU=14.90


BLEU eval pubhealth_misclassified_top200:   0%|          | 0/200 [00:00<?, ?it/s]

[pubhealth_misclassified_top200] scored=200 BLEU=1.75
BLEU after SFT:
Intel: 14.901753518333527
PubHealth: 1.7500258013308019
